In [1]:
from pyspark.sql import SparkSession

# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Scrap NYCTaxi") \
    .getOrCreate()

# Acceder al contexto de Spark (SparkContext)
sc = spark.sparkContext

In [2]:
import json
import time
import requests

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
URL_BASE = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
NOMBRE_ARCHIVO_LOCAL = "lista_urls_taxis_final.json"

# ==========================================
# 2. ESCANEO (Sincero e Inteligente)
# ==========================================
tareas = []
print("Iniciando escaneo... (Hola Amazon, somos Python, venimos en son de paz)")

for anio in range(2009, 2027):
    for mes in range(1, 13):
        for tipo in ["yellow", "green"]:

            # REGLA: Los taxis verdes nacieron en agosto de 2013
            if tipo == "green" and (anio < 2013 or (anio == 2013 and mes < 8)):
                continue

            archivo_parquet = f"{tipo}_tripdata_{anio}-{mes:02d}.parquet"
            url_completa = URL_BASE + archivo_parquet

            try:
                # Usamos requests normal, sin disfraces
                respuesta = requests.head(url_completa, timeout=10)

                if respuesta.status_code == 200:
                    peso_bytes = int(respuesta.headers.get('Content-Length', 0))
                    if peso_bytes > 0:
                        tareas.append({
                            "url_part": archivo_parquet,
                            "anio": str(anio),
                            "mes": f"{mes:02d}",
                            "tipo": tipo
                        })
                        print(f"[AÑADIDO] {archivo_parquet}")
                    else:
                        print(f"[OMITIDO] {archivo_parquet} (Vacío)")
                else:
                    print(f"[OMITIDO] {archivo_parquet} (No existe / {respuesta.status_code})")

            except Exception as e:
                print(f"[ERROR] {archivo_parquet}: {e}")

            # Micro-pausa
            time.sleep(0.1)

print(f"\n¡Finalizado! Total de archivos reales listos para descargar: {len(tareas)}")

Iniciando escaneo... (Hola Amazon, somos Python, venimos en son de paz)
[OMITIDO] yellow_tripdata_2009-01.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-02.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-03.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-04.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-05.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-06.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-07.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-08.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-09.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-10.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-11.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2009-12.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2010-01.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2010-02.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_2010-03.parquet (No existe / 403)
[OMITIDO] yellow_tripdata_20

In [3]:
spark.stop

<bound method SparkSession.stop of <pyspark.sql.session.SparkSession object at 0x7957a0573680>>